In [31]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
import itertools, tqdm

def read_params(path):
    params = pd.read_csv(path, header=None, sep=" ", index_col=False, lineterminator='\n')
    params.rename(columns={0:'img_name'}, inplace=True)
    params = params.set_index('img_name').T.to_dict('list')
    return params

def rotate_sh(inp_sh, n_step, axis):

    import pyshtools as pysh
    def toCoeff(c):
      t = pysh.SHCoeffs.from_zeros(2)
      t.set_coeffs(c[0], 0, 0)
      t.set_coeffs(c[1], 1, 1)
      t.set_coeffs(c[2], 1, -1)
      t.set_coeffs(c[3], 1, 0)
      t.set_coeffs(c[4], 2, -2)
      t.set_coeffs(c[5], 2, 1)
      t.set_coeffs(c[6], 2, -1)
      t.set_coeffs(c[7], 2, 2)
      t.set_coeffs(c[8], 2, 0)
      return t

    def toRGBCoeff(c):
      return [toCoeff(c[::3]), toCoeff(c[1::3]), toCoeff(c[2::3])]

    def toDeca(c):
      a = c.coeffs
      lst = [a[0, 0, 0],
             a[0, 1, 1],
             a[1, 1, 1],
             a[0, 1, 0],
             a[1, 2, 2],
             a[0, 2, 1],
             a[1, 2, 1],
             a[0, 2, 2],
             a[0, 2, 0]]
      return np.array(lst)

    def toRGBDeca(cc):
      return list(itertools.chain(*zip(toDeca(cc[0]), toDeca(cc[1]), toDeca(cc[2]))))

    def axisAngleToEuler(x, y, z, degree):
      xyz = np.array([x, y, z])
      xyz = xyz / np.linalg.norm(xyz)

      rot = R.from_mrp(xyz * np.tan(degree * np.pi / 180 / 4))
      return rot.as_euler('zyz', degrees=True)

    def rotateSH(sh_np, x, y, z, degree):
      cc = toRGBCoeff(sh_np)
      euler = axisAngleToEuler(x, y, z, degree)
      cc[0] = cc[0].rotate(*euler)
      cc[1] = cc[1].rotate(*euler)
      cc[2] = cc[2].rotate(*euler)
      return toRGBDeca(cc)
  
    if inp_sh.shape == (9, 3):
        inp_sh = inp_sh.flatten()  # [9, 3] -> [27,]
    n = n_step
    out_sh = []
    n = n_step
    for j in np.linspace(0, 360, n):
        moved = rotateSH(inp_sh, axis==0, axis==1, axis==2, j)
        sh_moved = np.array(moved)
        out_sh.append(sh_moved)

    out_sh = np.stack(out_sh, 0)    # [n_step, 27]
    return {'light':out_sh}

def sh_to_ld(sh):
    #NOTE: Roughly Convert the SH to light direction
    sh = sh.reshape(-1, 9, 3)
    ld = np.mean(sh[0:1, 1:4, :], axis=2)
    return ld
  
def batch_orth_proj(X, camera):
    ''' orthgraphic projection
        X:  3d vertices, [bz, n_point, 3]
        camera: scale and translation, [bz, 3], [scale, tx, ty]
    '''
    # camera = camera.clone().view(-1, 1, 3)
    camera = camera.copy().reshape(-1, 1, 3)
    X_trans = X[:, :, :2] + camera[:, :, 1:]        # Translation with x + tx, y + ty
    X_trans = np.concatenate([X_trans, X[:,:,2:]], 2)    # Concat with z : (x + tx, y + ty, z)
    shape = X_trans.shape
    Xn = (camera[:, :, 0:1] * X_trans) # Scaling
    return Xn

In [126]:
import numpy as np
import plotly.graph_objects as go

def normalize_rows(v, eps=1e-9):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n = np.maximum(n, eps)
    return v / n

def make_sphere(n_u=80, n_v=40):
    u = np.linspace(0, 2*np.pi, n_u)
    v = np.linspace(0, np.pi, n_v)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    return xs, ys, zs

def init_scene(title="Light Directions (Y-up, view +Z→-Z)"):
    xs, ys, zs = make_sphere()
    sphere = go.Surface(x=xs, y=ys, z=zs, showscale=False, opacity=0.15, name="unit sphere")

    fig = go.Figure(data=[sphere])
    fig.update_scenes(
        xaxis=dict(range=[-1,1], title='X'),
        yaxis=dict(range=[-1,1], title='Y'),
        zaxis=dict(range=[-1,1], title='Z'),
        aspectmode='cube'
    )
    fig.update_layout(
        scene_camera=dict(
            up=dict(x=0, y=1, z=0),     # Y-up
            eye=dict(x=0, y=0, z=2.5),  # look from +Z toward -Z
            center=dict(x=0, y=0, z=0)
        ),
        title=title,
        margin=dict(l=0, r=0, t=40, b=0),
        legend=dict(itemsizing='constant')
    )
    return fig

def add_light_traj(fig, light_dirs, name="traj", colorscale="Viridis",
                   show_start=True, show_end=True, line_color="rgba(120,120,120,0.6)"):
    D = normalize_rows(np.asarray(light_dirs, dtype=float))
    N = len(D)

    # time-colored points + gray line
    fig.add_trace(go.Scatter3d(
        x=D[:,0], y=D[:,1], z=D[:,2],
        mode="markers+lines",
        marker=dict(size=4, color=np.arange(N), colorscale=colorscale, showscale=False),
        line=dict(color=line_color, width=1),
        name=name,
        legendgroup=name,
        showlegend=True
    ))
    # start/end markers
    if N and show_start:
        fig.add_trace(go.Scatter3d(
            x=[D[0,0]], y=[D[0,1]], z=[D[0,2]],
            mode="markers",
            marker=dict(size=8, color="red", symbol="diamond"),
            name=f"{name} start",
            legendgroup=name,
            showlegend=True
        ))
    if N and show_end:
        fig.add_trace(go.Scatter3d(
            x=[D[-2,0]], y=[D[-2,1]], z=[D[-2,2]],
            mode="markers",
            marker=dict(size=8, color="green", symbol="cross"),
            name=f"{name} end",
            legendgroup=name,
            showlegend=True
        ))
    return fig

# ---- Usage ----
# fig = init_scene()
# fig = add_light_traj(fig, light_source, name="IC-Light yaw=10°")
# fig = add_light_traj(fig, other_light_source, name="Our method roll=15°", colorscale="Plasma")
# fig.show()


In [129]:
light_path = f'/data/mint/DPM_Dataset/ffhq_256_with_anno/params/valid/ffhq-valid-light-anno.txt'

light_params = read_params(light_path)
cam_params = read_params(light_path.replace('light', 'cam'))
img_id = '64240.jpg'
# img_id = '65112.jpg'
# img_id = '60065.jpg'
# img_id = '64036.jpg'
light = np.array(light_params[img_id])
cam = np.array(cam_params[img_id])

def get_rotated_ld(axis, light, cam, n_step=60):
    rotated_sh = rotate_sh(light, n_step, axis=axis)['light'] # [n_step, 27]
    light_source = []
    for i in range(rotated_sh.shape[0]):
        ld = sh_to_ld(sh=np.array(rotated_sh[i]))
        ld = batch_orth_proj(ld[None, ...], cam[None, ...])     # This fn takes pts=Bx3, cam=Bx3
        ld[:, :, 1:] = -ld[:, :, 1:]
        ray = ld.reshape(3)
        ray = ray / np.linalg.norm(ray)
        if axis == 1: ray[2] *= -1
        light_source.append(ray)
    light_source = np.stack(light_source, 0)  # [n_step, 3]
    return light_source

fig = init_scene()
ld_axis1 = get_rotated_ld(axis=1, light=light, cam=cam, n_step=30)
ld_axis1[:, 0] *= -1
fig = add_light_traj(fig, ld_axis1, name="ld_axis1")
ld_axis2 = get_rotated_ld(axis=2, light=light, cam=cam, n_step=30)
ld_axis2[:, 0] *= -1
fig = add_light_traj(fig, ld_axis2, name="ld_axis2", colorscale="Plasma")
fig.show()

/tmp/ipykernel_876537/2336361822.py:52: UserWarning:

Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.

